Adapted from https://github.com/gifale95/eeg_encoding/tree/main

In [ ]:
from pathlib import Path
from typing import List, Dict, Tuple, Union
from collections import defaultdict
import json


import numpy as np
import pandas as pd
import h5py
import mne
from scipy import io
from tqdm.auto import tqdm

from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
from helpers import compute_ceiling_splithalf, compute_ceiling_variancebased

In [ ]:
SUBJECT_IDS = list(range(1, 11))
SUBJECTS = ["sub-{:02d}".format(i) for i in SUBJECT_IDS]

TIME_POINTS = 0.0, 0.8  # Available: -0.2 to 0.8

ROIS = {
    "occipital": "O",
    "parietal": "P",
    "temporal": "T",
    "frontal": "F",
    "central": "C",
}

ROIS_EXTENDED = [
    "occipital",
    "parietal",
    "temporal",
    "frontal",
    "central",
    "occipital_parietal",
    "whole_brain"
]

SUBJECTS, TIME_POINTS, ROIS

In [ ]:
ds_dir = "${MBS_THINGS_RAW_DIR}/things_eeg2/source_data"
ds_dir = Path(ds_dir)
assert ds_dir.exists(), f"Dataset directory {ds_dir} does not exist."

In [ ]:
def load_session_raw_and_events(project_dir, sub, ses, partition):
    """
    Load EEG+behavior for a single subject/session/partition from source files
    and build an events array with correct event times and image IDs.

    Returns
    -------
    raw : mne.io.Raw
        Continuous EEG for the full session (all parts concatenated).
    events : np.ndarray, shape (n_events, 3)
        MNE-style events: [sample_idx, 0, event_id] where event_id is image ID
        (targets 0 are mapped to 99999).
    """
    # How many parts?
    if partition == "train":
        n_parts = 5
    elif partition == "test":
        n_parts = 1
    else:
        raise ValueError(f"Unknown partition: {partition}")

    base_dir = (
        Path(project_dir)
        / f"sub-{sub:02d}"
        / f"ses-{ses:02d}"
    )

    raw_parts = []
    all_events = []
    offset = 0  # cumulative sample offset across parts

    for p in range(n_parts):
        if partition == "train":
            beh_path = (
                base_dir
                / "beh"
                / f"sub-{sub:02d}_ses-{ses:02d}_task-{partition}_part-{p+1:02d}_beh.mat"
            )
            eeg_path = (
                base_dir
                / "eeg"
                / f"sub-{sub:02d}_ses-{ses:02d}_task-{partition}_part-{p+1:02d}_eeg.vhdr"
            )
        else:  # test
            beh_path = (
                base_dir
                / "beh"
                / f"sub-{sub:02d}_ses-{ses:02d}_task-{partition}_beh.mat"
            )
            eeg_path = (
                base_dir
                / "eeg"
                / f"sub-{sub:02d}_ses-{ses:02d}_task-{partition}_eeg.vhdr"
            )

        # --- Load behavior ---
        beh_data = io.loadmat(beh_path)["data"]
        events_beh = beh_data[0][0][2]["tot_img_number"][0]

        # Make sure it's a flat 1D array of ints
        events_beh = np.asarray(events_beh, dtype=int).ravel()

        # Change target events from 0 to 99999
        events_beh[events_beh == 0] = 99999

        # --- Load EEG ---
        raw_part = mne.io.read_raw_brainvision(str(eeg_path), preload=True)

        # Original code: get event samples from annotations
        events_mne, _ = mne.events_from_annotations(raw_part)
        # and keep ONLY the first column (sample indices)
        events_samples = events_mne[:, 0].astype(int)

        if events_samples.shape[0] != events_beh.shape[0]:
            raise RuntimeError(
                f"Length mismatch in sub {sub}, ses {ses}, part {p+1}, {partition}: "
                f"{events_samples.shape[0]} EEG events vs "
                f"{events_beh.shape[0]} behavioral events."
            )

        # Build proper (N, 3) event matrix: [sample, 0, image_id]
        part_events = np.column_stack(
            [
                events_samples + offset,
                np.zeros(events_samples.shape[0], dtype=int),
                events_beh,
            ]
        )

        all_events.append(part_events)
        offset += raw_part.n_times
        raw_parts.append(raw_part)

    # Concatenate EEG across parts
    raw = mne.concatenate_raws(raw_parts)
    events = np.concatenate(all_events, axis=0)

    # Keep only EEG channels (drop non-EEG)
    picks_eeg = mne.pick_types(raw.info, eeg=True, exclude=[])
    raw.pick(picks_eeg)

    # Sanity check for shape
    if events.ndim != 2 or events.shape[1] != 3:
        raise RuntimeError(f"Events must be (N, 3); got {events.shape}.")

    return raw, events

In [ ]:
def epoch_and_sort_session(
    raw,
    events,
    data_part,
    sfreq_target=100,
    # channel_pattern=r"^O *|^P *",
    channel_pattern=None,
    seed=0,
    tmin=-0.2,
    tmax=0.8,
):
    """
    Epoch one session and reorder trials into
    [image_cond x repetitions x channels x time].

    Parameters
    ----------
    raw : mne.io.Raw
        Continuous EEG.
    events : np.ndarray
        MNE-style events, event_id = image condition IDs (99999 = targets).
    data_part : {'train', 'test'}
    sfreq_target : float
        Target sampling frequency for resampling.
    channel_pattern : str or None
        Regex for channels to keep. If None, keep all EEG channels.
    seed : int
        Random seed for trial selection / shuffling.
    tmin, tmax : float
        Epoch window in seconds.

    Returns
    -------
    sorted_data : np.ndarray
        [n_conditions x max_rep x n_channels x n_times]
    img_cond : np.ndarray
        Unique image condition IDs in the order of the first axis.
    ch_names : list of str
        Channel names.
    times : np.ndarray
        Time points in seconds.
    """
    # Channel selection
    if channel_pattern is not None:
        chan_idx = np.asarray(
            mne.pick_channels_regexp(raw.info["ch_names"], channel_pattern)
        )
        new_chans = [raw.info["ch_names"][c] for c in chan_idx]
        raw = raw.copy().pick_channels(new_chans)

    # Remove target trials
    non_target_mask = events[:, 2] != 99999
    events = events[non_target_mask]

    # Epoch, baseline correct, and resample
    epochs = mne.Epochs(
        raw,
        events,
        tmin=tmin,
        tmax=tmax,
        baseline=(None, 0),
        preload=True,
    )

    if sfreq_target is not None and sfreq_target < raw.info["sfreq"]:
        epochs.resample(sfreq_target)
    
    # Crop to desired time points
    epochs = epochs.crop(tmin=TIME_POINTS[0], tmax=TIME_POINTS[1])

    data = epochs.get_data(units='uV')          # [n_trials, n_channels, n_times]
    trial_labels = epochs.events[:, 2]
    img_cond = np.unique(trial_labels)

    ch_names = epochs.info["ch_names"]
    times = epochs.times

    # Max repetitions per condition
    if data_part == "test":
        max_rep = 20
    else:
        max_rep = 2

    sorted_data = np.zeros(
        (len(img_cond), max_rep, data.shape[1], data.shape[2]),
        dtype=data.dtype,
    )

    for i, cond in enumerate(img_cond):
        idx = np.where(trial_labels == cond)[0]
        # print(f"Condition {cond}: found {len(idx)} trials.")
        # Randomly pick max_rep trials
        rng = np.random.default_rng(seed)
        idx = rng.choice(idx, size=max_rep, replace=False)
        sorted_data[i] = data[idx]

    return sorted_data, img_cond, ch_names, times

In [ ]:
def merge_sessions(epoched_list, img_cond_list, reps=4):
    """
    Merge data across sessions, grouping by image condition.

    Parameters
    ----------
    epoched_list : list of np.ndarray
        Each array: [n_cond_session x max_rep x C x T].
    img_cond_list : list of np.ndarray
        Conditions array per session, length n_cond_session.
    reps : int
        Number of repetitions to collect per condition.

    Returns
    -------
    merged_data : np.ndarray
        [n_unique_cond x n_total_rep_per_cond x C x T]
    img_cond_unique : np.ndarray
        Sorted unique condition IDs.
    """
    # Stack over "condition" axis
    all_data = np.concatenate(epoched_list, axis=0)      # [sum n_cond_s, max_rep, C, T]
    all_cond = np.concatenate(img_cond_list, axis=0)     # [sum n_cond_s]

    img_cond_unique = np.unique(all_cond)
    max_rep = epoched_list[0].shape[1]
    n_ses = len(epoched_list)
    C = all_data.shape[2]
    T = all_data.shape[3]

    merged_train = np.zeros(
        (len(img_cond_unique), reps, C, T),
        dtype=all_data.dtype,
    )
    # print("merged_train shape:", merged_train.shape)
    

    for i, cond in enumerate(img_cond_unique):
        idx = np.where(all_cond == cond)[0]  # indices of this cond across sessions
        # Collect data from all sessions for this condition
        cond_chunks = [all_data[j] for j in idx]  # each: [max_rep x C x T]
        cond_data = np.concatenate(cond_chunks, axis=0)  # [n_ses*max_rep, C, T]
        # print("cond_data shape:", cond_data.shape)
        merged_train[i] = cond_data

    return merged_train, img_cond_unique

In [ ]:
def process_subject(
    ds_dir,
    sub,
    n_ses,
    sfreq_target=100,
    # channel_pattern=r"^O *|^P *",
    channel_pattern=None,
    seed=42,
):
    """
    Process one subject: load source, build events, epoch, reorder,
    and merge sessions for both train and test.

    Returns
    -------
    train_responses : np.ndarray
        [n_train_images x n_train_reps x C x T]
    train_img_indices : np.ndarray
        Image IDs for train first axis.
    test_responses : np.ndarray
        [n_test_images x n_test_reps x C x T]
    test_img_indices : np.ndarray
        Image IDs for test first axis.
    ch_names : list of str
    times : np.ndarray
    """
    # ---- TRAIN ----
    train_epoched_per_ses = []
    train_img_cond_per_ses = []
    ch_names = None
    times = None

    for ses in range(1, n_ses + 1):
        raw_train, events_train = load_session_raw_and_events(
            ds_dir, sub, ses, partition="train"
        )
        sorted_train, img_cond_train, ch_names, times = epoch_and_sort_session(
            raw_train,
            events_train,
            data_part="train",
            sfreq_target=sfreq_target,
            channel_pattern=channel_pattern,
            seed=seed,
        )
        # print(f"Session {ses}: train data shape {sorted_train.shape}")
        # print(f"Session {ses}: train image conditions {img_cond_train}")
        train_epoched_per_ses.append(sorted_train)
        train_img_cond_per_ses.append(img_cond_train)

    train_responses, train_img_indices = merge_sessions(
        train_epoched_per_ses,
        train_img_cond_per_ses,
        reps=4,
    )

    # ---- TEST ----
    test_epoched_per_ses = []
    test_img_cond_per_ses = []
    test_img_cond = None

    for ses in range(1, n_ses + 1):
        raw_test, events_test = load_session_raw_and_events(
            ds_dir, sub, ses, partition="test"
        )
        sorted_test, img_cond_test, ch_names, times = epoch_and_sort_session(
            raw_test,
            events_test,
            data_part="test",
            sfreq_target=sfreq_target,
            channel_pattern=channel_pattern,
            seed=seed,
        )
        test_epoched_per_ses.append(sorted_test)
        test_img_cond_per_ses.append(img_cond_test)
        if test_img_cond is None:
            test_img_cond = img_cond_test
        else:
            # sanity check: conditions should match across sessions
            assert np.array_equal(
                test_img_cond, img_cond_test
            ), "Test conditions differ across sessions!"

    test_responses, test_img_indices = merge_sessions(epoched_list=test_epoched_per_ses, img_cond_list=test_img_cond_per_ses, reps=80)

    return (
        train_responses,
        train_img_indices,
        test_responses,
        test_img_indices,
        ch_names,
        times,
    )

In [ ]:
# raw, events = load_session_raw_and_events(
#     ds_dir,
#     sub=1,
#     ses=1,
#     partition="train",
# )
# epoched_data, img_cond, ch_names, times = epoch_and_sort_session(
#     raw,
#     events,
#     data_part="train",
#     sfreq_target=250,
#     # channel_pattern=r"^O *|^P *",
#     channel_pattern=None,
#     seed=0,
#     tmin=-0.2,
#     tmax=0.8,
# )

In [ ]:
train_responses_by_subject = {}
test_responses_by_subject = {}

train_img_indices = None
test_img_indices = None
common_ch_names = None
common_times = None

for sub in tqdm(SUBJECT_IDS, desc="Processing subjects"):
    (
        train_resp,
        train_idx,
        test_resp,
        test_idx,
        ch_names,
        times,
    ) = process_subject(
        ds_dir=ds_dir,
        sub=sub,
        n_ses=4,
        sfreq_target=100,
        channel_pattern=None,
        seed=42,
    )

    subj = f"sub-{sub:02d}"
    train_responses_by_subject[subj] = train_resp
    test_responses_by_subject[subj] = test_resp

    if train_img_indices is None:
        train_img_indices = train_idx
        test_img_indices = test_idx
        common_ch_names = ch_names
        common_times = times
    else:
        # Sanity check: same stimuli across subjects
        assert np.array_equal(
            train_img_indices, train_idx
        ), "Train image indices differ across subjects!"
        assert np.array_equal(
            test_img_indices, test_idx
        ), "Test image indices differ across subjects!"

In [ ]:
channels_by_roi = {}
channels_by_roi_masks = {}
for roi_name in tqdm(ROIS_EXTENDED):
    if roi_name == "whole_brain":
        channels_by_roi[roi_name] = common_ch_names
        channels_by_roi_masks[roi_name] = np.ones_like(common_ch_names, dtype=bool)
    elif roi_name == "occipital_parietal":
        channels_by_roi[roi_name] = [
            ch for ch in common_ch_names if (ch.startswith("O") or ch.startswith("P"))
        ]
        channels_by_roi_masks[roi_name] = np.where(
            np.array([(ch.startswith("O") or ch.startswith("P")) for ch in common_ch_names])
        )[0]
    else:
        roi_prefix = ROIS[roi_name]
        channels_by_roi[roi_name] = [
            ch for ch in common_ch_names if ch.startswith(roi_prefix)
        ]
        channels_by_roi_masks[roi_name] = np.where(
            np.array([ch.startswith(roi_prefix) for ch in common_ch_names])
        )[0]
    
    print(f"ROI {roi_name}: {len(channels_by_roi[roi_name])} channels.")
    
remaining_channels = set(common_ch_names)
for ch_list in channels_by_roi.values():
    remaining_channels -= set(ch_list)
print(f"Channels not assigned to any ROI: {remaining_channels}")


In [ ]:
train_responses_by_subject_roi = {}

for subj, subj_data in tqdm(train_responses_by_subject.items()):
    train_responses_by_subject_roi[subj] = {}
    for roi_name, ch_mask in tqdm(channels_by_roi_masks.items(), leave=False):
        
        # Select data for these channels
        train_data = subj_data[:, :, ch_mask, :]
        
        # Stimuli x Repetition x Channels x Time points -> Stimuli x Channels x Time points x Repetition 
        train_data = np.transpose(train_data, (0, 2, 3, 1))
        
        train_responses_by_subject_roi[subj][roi_name] = train_data
        
        
test_responses_by_subject_roi = {}
for subj, subj_data in tqdm(test_responses_by_subject.items()):
    test_responses_by_subject_roi[subj] = {}
    for roi_name, ch_mask in tqdm(channels_by_roi_masks.items(), leave=False):
        
        # Select data for these channels
        test_data = subj_data[:, :, ch_mask, :]
        
        # Stimuli x Repetition x Channels x Time points -> Stimuli x Channels x Time points x Repetition 
        test_data = np.transpose(test_data, (0, 2, 3, 1))
        
        test_responses_by_subject_roi[subj][roi_name] = test_data
        


In [ ]:
for roi_name in ROIS_EXTENDED:
    train_data = train_responses_by_subject_roi[SUBJECTS[0]][roi_name]
    test_data = test_responses_by_subject_roi[SUBJECTS[0]][roi_name]
    
    print(f"ROI {roi_name}, train data shape: {train_data.shape}, test data shape: {test_data.shape}")

In [ ]:
noise_ceilings_train_variancebased, noise_ceilings_train_splithalf = {}, {}
noise_ceilings_test_variancebased, noise_ceilings_test_splithalf = {}, {}

for subj in tqdm(SUBJECTS):
    noise_ceilings_train_variancebased[subj] = {}
    noise_ceilings_train_splithalf[subj] = {}
    noise_ceilings_test_variancebased[subj] = {}
    noise_ceilings_test_splithalf[subj] = {}

    for roi_name in tqdm(ROIS_EXTENDED, total=len(ROIS_EXTENDED), desc=f"Subject {subj} ROIs", leave=False):
        responses_train = train_responses_by_subject_roi[subj][roi_name]  # Stimuli x Channels x Time points x Repetition
        responses_test = test_responses_by_subject_roi[subj][roi_name]  # Stimuli x Channels x Time points x Repetition

        n_stimuli_train, n_channels, n_timepoints, n_repetitions_train = responses_train.shape
        n_stimuli_test, n_channels, n_timepoints, n_repetitions_test = responses_test.shape

        # Permute to (Channels,  Time points, Stimuli, Repetition)
        reshaped_responses_train = np.transpose(responses_train, (1, 2, 0, 3))
        reshaped_responses_test = np.transpose(responses_test, (1, 2, 0, 3))

        # Compute noise ceilings
        noise_ceilings_train_variancebased[subj][roi_name] = compute_ceiling_variancebased(reshaped_responses_train, nan_policy='omit')  # (Channels * Time points,)
        noise_ceilings_test_variancebased[subj][roi_name] = compute_ceiling_variancebased(reshaped_responses_test, nan_policy='omit')  # (Channels * Time points,)


        noise_ceilings_train_splithalf[subj][roi_name] = compute_ceiling_splithalf(reshaped_responses_train).mean(-1)  # (Channels * Time points,)
        noise_ceilings_test_splithalf[subj][roi_name] = compute_ceiling_splithalf(reshaped_responses_test).mean(-1)  # (Channels * Time points,)
        

        

In [ ]:
np.abs(noise_ceilings_train_variancebased[subj]['occipital_parietal'] - noise_ceilings_train_splithalf[subj]['occipital_parietal']).max()

In [ ]:
# visualize

fig, axes = plt.subplots(5, 2, figsize=(20, 20), dpi=100)

for i, subj in enumerate(SUBJECTS):
    ax = axes[i // 2, i % 2]
    nc_train_vb = noise_ceilings_train_variancebased[subj]['occipital_parietal']  # (Channels * Time points,)
    nc_test_vb = noise_ceilings_test_variancebased[subj]['occipital_parietal']  # (Channels * Time points,)

    nc_train_sh = noise_ceilings_train_splithalf[subj]['occipital_parietal']  # (Channels * Time points,)
    nc_test_sh = noise_ceilings_test_splithalf[subj]['occipital_parietal']  # (Channels * Time points,)

    time = np.linspace(TIME_POINTS[0], TIME_POINTS[1], nc_train_vb.shape[1])

    sns.lineplot(x=time, y=nc_train_vb.mean(0), ax=ax, label='Train (Variance-based)')
    sns.lineplot(x=time, y=nc_train_sh.mean(0), ax=ax, label='Train (Split-half)')
    sns.lineplot(x=time, y=nc_test_vb.mean(0), ax=ax, label='Test (Variance-based)')
    sns.lineplot(x=time, y=nc_test_sh.mean(0), ax=ax, label='Test (Split-half)')
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Noise Ceiling (Pearson r)')
    ax.set_title(f"Subject {subj}", fontsize=16, fontweight='bold')

    # put a horizontal line at maximum of of all ceilings
    max_ceiling_test = max(nc_test_vb.mean(0).max(), nc_test_sh.mean(0).max())
    max_ceiling_train = max(nc_train_vb.mean(0).max(), nc_train_sh.mean(0).max())
    ax.axhline(max_ceiling_test, color='k', linestyle='--', label='Max Ceiling (Test) @ {:.2f}'.format(max_ceiling_test))
    ax.axhline(max_ceiling_train, color='k', linestyle=':', label='Max Ceiling (Train) @ {:.2f}'.format(max_ceiling_train))
    ax.axhline(0, color='k', linestyle=':')

    ax.legend()
    
plt.tight_layout()

In [ ]:
    # visualize
for roi_name in ROIS_EXTENDED:

    fig, axes = plt.subplots(5, 2, figsize=(20, 20), dpi=100)

    for i, subj in enumerate(SUBJECTS):
        ax = axes[i // 2, i % 2]
        nc_train_vb = noise_ceilings_train_variancebased[subj][roi_name]  # (Channels * Time points,)
        nc_test_vb = noise_ceilings_test_variancebased[subj][roi_name]  # (Channels * Time points,)

        nc_train_sh = noise_ceilings_train_splithalf[subj][roi_name]  # (Channels * Time points,)
        nc_test_sh = noise_ceilings_test_splithalf[subj][roi_name]  # (Channels * Time points,)

        time = np.linspace(TIME_POINTS[0], TIME_POINTS[1], nc_train_vb.shape[1])

        sns.lineplot(x=time, y=nc_train_vb.mean(0), ax=ax, label='Train (Variance-based)')
        sns.lineplot(x=time, y=nc_train_sh.mean(0), ax=ax, label='Train (Split-half)')
        sns.lineplot(x=time, y=nc_test_vb.mean(0), ax=ax, label='Test (Variance-based)')
        sns.lineplot(x=time, y=nc_test_sh.mean(0), ax=ax, label='Test (Split-half)')
        ax.set_xlabel('Time (s)')
        ax.set_ylabel('Noise Ceiling (Pearson r)')
        ax.set_title(f"Subject {subj}", fontsize=16, fontweight='bold')

        # put a horizontal line at maximum of of all ceilings
        max_ceiling_test = max(nc_test_vb.mean(0).max(), nc_test_sh.mean(0).max())
        max_ceiling_train = max(nc_train_vb.mean(0).max(), nc_train_sh.mean(0).max())
        ax.axhline(max_ceiling_test, color='k', linestyle='--', label='Max Ceiling (Test) @ {:.2f}'.format(max_ceiling_test))
        ax.axhline(max_ceiling_train, color='k', linestyle=':', label='Max Ceiling (Train) @ {:.2f}'.format(max_ceiling_train))
        ax.axhline(0, color='k', linestyle=':')

        ax.legend()
    
    plt.suptitle(f"ROI: {roi_name}", fontsize=20, fontweight='bold')
    plt.tight_layout()

In [ ]:
# subject_responses_train_avg = {
#     subj: {
#         roi: train_responses_by_subject_roi[subj][roi].mean(-1)  # Stimuli x Channels x Time points
#         for roi in ROIS_EXTENDED
#     }
#     for subj in SUBJECTS
# }

# subject_responses_test_avg = {
#     subj: {
#         roi: test_responses_by_subject_roi[subj][roi].mean(-1)  # Stimuli x Channels x Time points
#     for roi in ROIS_EXTENDED
#     }
#     for subj in SUBJECTS
# }


subject_responses_train_avg, subject_responses_test_avg = {}, {}

for subj in tqdm(SUBJECTS):
    subject_responses_train_avg[subj] = {}
    subject_responses_test_avg[subj] = {}
    
    for roi_name in tqdm(ROIS_EXTENDED, leave=False):
        train_data = train_responses_by_subject_roi[subj][roi_name].mean(-1)  # Stimuli x Channels x Time points
        test_data = test_responses_by_subject_roi[subj][roi_name].mean(-1)  # Stimuli x Channels x Time points
        
        subject_responses_train_avg[subj][roi_name] = train_data
        subject_responses_test_avg[subj][roi_name] = test_data
        
        if subj == SUBJECTS[-1]:
            print(f"ROI {roi_name}: {train_data.shape}")
            print(f"ROI {roi_name}: {test_data.shape}")
    

In [ ]:
subject_responses_train_avg[SUBJECTS[0]][ROIS_EXTENDED[0]].shape, subject_responses_test_avg[SUBJECTS[0]][ROIS_EXTENDED[0]].shape

In [ ]:
noise_ceilings_train_variancebased[subj][ROIS_EXTENDED[0]].shape, noise_ceilings_test_variancebased[subj][ROIS_EXTENDED[0]].shape

In [ ]:

image_metadata = Path(ds_dir).parent / "eeg2_preprocessed/image_metadata.npy"
image_metadata = np.load(image_metadata, allow_pickle=True, fix_imports=True).item()

stimuli_files_train = image_metadata['train_img_files']
stimuli_files_test = image_metadata['test_img_files']

train_stimulus_ids = [f"{stimulus.rsplit('_', 1)[0]}/{stimulus}" for stimulus in stimuli_files_train]
test_stimulus_ids = [f"{stimulus.rsplit('_', 1)[0]}/{stimulus}" for stimulus in stimuli_files_test]

len(train_stimulus_ids), len(test_stimulus_ids)

### Concatenate data

In [ ]:
processed_data = {
    "train" :
        {
            "stimulus_ids": train_stimulus_ids,
            "neural_data": subject_responses_train_avg,
        },
    "test" :
        {
            "stimulus_ids": test_stimulus_ids,
            "neural_data": subject_responses_test_avg,
        },
    "noise_ceilings": noise_ceilings_test_variancebased,
    "noise_ceilings_train": noise_ceilings_train_variancebased,
}

### Metadata

In [ ]:
metadata = {
    "desc": """
    The neural data is from the THINGS EEG 2, recorded from 10 humans.
    All neural data is averaged across trials for each image.
    The neural data is concatenated across subjects and channels.
    """
}
metadata_str = json.dumps(metadata, indent=2)
metadata_str = json.dumps(metadata, indent=2).encode('utf-8')

### Save to disk

In [ ]:
data_dir = '${MBS_DATA_PREP_OUTPUT_DIR}'
filename = f'things_eeg2.h5'

data_dir = Path(data_dir)
data_path = data_dir / filename

if not data_dir.exists():
    data_dir.mkdir(parents=False, exist_ok=False)

In [ ]:
with h5py.File(data_path, 'w') as f:
    for split in ['train', 'test']:
        f.create_dataset(f"{split}/stimulus_ids", data=processed_data[split]['stimulus_ids'])

        for subj in tqdm(SUBJECTS, desc="Subject responses"):
            for roi_name in tqdm(ROIS_EXTENDED, leave=False):
                f.create_dataset(f"{split}/neural_data/{subj}/{roi_name}", data=processed_data[split]['neural_data'][subj][roi_name])
                
    for subj in tqdm(SUBJECTS, desc="Subject noise ceilings"):
        for roi_name in tqdm(ROIS_EXTENDED, leave=False):
            f.create_dataset(f"noise_ceilings/{subj}/{roi_name}", data=processed_data['noise_ceilings'][subj][roi_name])
            f.create_dataset(f"noise_ceilings_train/{subj}/{roi_name}", data=processed_data['noise_ceilings_train'][subj][roi_name])

    f.attrs['metadata'] = metadata_str
    f.attrs['rois'] = ROIS_EXTENDED
    f.attrs['subjects'] = list(SUBJECTS)
    f.attrs['splits'] = ['train', 'test']
    f.attrs['max_nc'] = 100
    f.attrs['time_points'] = common_times.tolist()
    f.attrs['channels'] = common_ch_names
    
    f.close()


In [ ]:
loaded_data = defaultdict(dict)
with h5py.File(data_path, 'r') as f:
    splits = f.attrs['splits']
    subjects = f.attrs['subjects']
    rois = f.attrs['rois']
    channels = f.attrs['channels']
    for split in splits:
        loaded_data[split]['stimulus_ids'] = f[split]['stimulus_ids'][()]
        
        loaded_data[split]['neural_data'] = {}
        for subj in subjects:
            loaded_data[split]['neural_data'][subj] = {}
            for roi in rois:
                loaded_data[split]['neural_data'][subj][roi] = f[split]['neural_data'][subj][roi][()]
                
    for subj in subjects:
        loaded_data['noise_ceilings'][subj] = {}
        for roi in rois:
            loaded_data['noise_ceilings'][subj][roi] = f['noise_ceilings'][subj][roi][()]
            
        loaded_data['noise_ceilings_train'][subj] = {}
        for roi in rois:
            loaded_data['noise_ceilings_train'][subj][roi] = f['noise_ceilings_train'][subj][roi][()]


In [ ]:
len(rois), len(subjects), len(splits)

In [ ]:
loaded_data['test'].keys()
loaded_data['test']['neural_data']['sub-01']['whole_brain'].shape, loaded_data['noise_ceilings']['sub-01']['whole_brain'].shape

In [ ]:
data_dir = Path('${MBS_DATA_PREP_OUTPUT_DIR}')
filename_1 = f'things_eeg2.h5'
filename_2 = f'things_eeg2_old.h5'

atol = 1e-6
diffs = []
diffs_nc_vb = []
diffs_nc_sp = []
with h5py.File(data_dir / filename_1, 'r') as f1, h5py.File(data_dir / filename_2, 'r') as f2:
    for split in tqdm(f1.attrs['splits']):
        for subj in tqdm(f1.attrs['subjects'], leave=False):
            for roi in tqdm(f1.attrs['rois'], leave=False):
                data1 = f1[f"{split}/neural_data/{subj}/{roi}"][()]
                data2 = f2[f"{split}/neural_data/{subj}/{roi}"][()] * 1.0e6  # old data in V, new data in uV
                diffs.append(np.abs(data1 - data2).sum())
                assert np.isclose(data1, data2, atol=atol).all(), f"Data mismatch in {split}, {subj}, {roi}"
    
    for subj in tqdm(f1.attrs['subjects'], leave=False):
        for roi in tqdm(f1.attrs['rois'], leave=False):
            nc1 = f1[f"noise_ceilings/{subj}/{roi}"][()]
            nc2 = f2[f"noise_ceilings/{subj}/{roi}"][()]
            diffs_nc_vb.append(np.abs(nc1 - nc2).sum())
            assert np.isclose(nc1, nc2, atol=atol).all(), f"Noise ceiling mismatch in {subj}, {roi}"
            
            nc1_train = f1[f"noise_ceilings_train/{subj}/{roi}"][()]
            nc2_train = f2[f"noise_ceilings_train/{subj}/{roi}"][()]
            diffs_nc_sp.append(np.abs(nc1_train - nc2_train).sum())
            assert np.isclose(nc1_train, nc2_train, atol=atol).all(), f"Noise ceiling train mismatch in {subj}, {roi}"
            
print("Max difference between datasets:", np.max(diffs), np.max(diffs_nc_vb), np.max(diffs_nc_sp))
print("Total difference between datasets:", np.sum(diffs), np.sum(diffs_nc_vb), np.sum(diffs_nc_sp))